# Fase 0: Exploratory Data Analysis

Este notebook valida la disponibilidad de los datos, resume la distribución de clases y visualiza ejemplos de CXR y CT antes de iniciar los experimentos de clasificación.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib_cache'))

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

from src.config import config
from src.data.preprocessing import get_cxr_dataframes

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 20)

PROJECT_ROOT

## 1. Disponibilidad de datasets

In [ ]:
cxr_root = config.CXR_DIR
ct_root = config.CT_DIR

print(f'CXR root: {cxr_root} | exists={cxr_root.exists()}')
print(f'CT root:  {ct_root} | exists={ct_root.exists()}')

cxr_classes = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
ct_classes = ['CT-0', 'CT-1', 'CT-2', 'CT-3', 'CT-4']

## 2. Distribución CXR

In [ ]:
cxr_counts = []
for cls in cxr_classes:
    image_dir = cxr_root / cls / 'images'
    mask_dir = cxr_root / cls / 'masks'
    cxr_counts.append({
        'class': cls,
        'images': len(list(image_dir.glob('*.png'))),
        'lung_masks': len(list(mask_dir.glob('*.png'))),
    })

cxr_counts_df = pd.DataFrame(cxr_counts)
cxr_counts_df['image_pct'] = 100 * cxr_counts_df['images'] / cxr_counts_df['images'].sum()
display(cxr_counts_df)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=cxr_counts_df, x='class', y='images', ax=ax)
ax.set_title('Distribución de clases CXR')
ax.set_xlabel('Clase')
ax.set_ylabel('Número de imágenes')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()

## 3. Split estratificado CXR

In [ ]:
cxr_train_df, cxr_val_df, cxr_test_df = get_cxr_dataframes(config.CXR_DIR, config.RANDOM_SEED)

split_summary = pd.concat([
    cxr_train_df.assign(split='train'),
    cxr_val_df.assign(split='val'),
    cxr_test_df.assign(split='test'),
]).groupby(['split', 'label']).size().unstack(fill_value=0)

display(split_summary)
display(split_summary.div(split_summary.sum(axis=1), axis=0).round(3))

## 4. Ejemplos CXR por clase

In [ ]:
fig, axes = plt.subplots(2, len(cxr_classes), figsize=(14, 6))
for col, cls in enumerate(cxr_classes):
    image_path = sorted((cxr_root / cls / 'images').glob('*.png'))[0]
    mask_path = sorted((cxr_root / cls / 'masks').glob('*.png'))[0]
    axes[0, col].imshow(Image.open(image_path).convert('L'), cmap='gray')
    axes[0, col].set_title(cls)
    axes[0, col].axis('off')
    axes[1, col].imshow(Image.open(mask_path).convert('L'), cmap='gray')
    axes[1, col].set_title('Máscara pulmón')
    axes[1, col].axis('off')
plt.tight_layout()

## 5. Distribución CT y fusión CT-3+

In [ ]:
ct_counts = []
for cls in ct_classes:
    cls_dir = ct_root / 'studies' / cls
    ct_counts.append({'class': cls, 'studies': len(list(cls_dir.glob('*.nii*')))})

ct_counts_df = pd.DataFrame(ct_counts)
ct_counts_df['merged_class'] = ct_counts_df['class'].replace({'CT-3': 'CT-3+', 'CT-4': 'CT-3+'})
ct_merged_df = ct_counts_df.groupby('merged_class', as_index=False)['studies'].sum()
ct_counts_df['pct'] = 100 * ct_counts_df['studies'] / ct_counts_df['studies'].sum()
ct_merged_df['pct'] = 100 * ct_merged_df['studies'] / ct_merged_df['studies'].sum()

display(ct_counts_df)
display(ct_merged_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=ct_counts_df, x='class', y='studies', ax=axes[0])
axes[0].set_title('CT original: 5 clases')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Estudios')

sns.barplot(data=ct_merged_df, x='merged_class', y='studies', ax=axes[1])
axes[1].set_title('CT tras fusionar CT-3 y CT-4')
axes[1].set_xlabel('Clase')
axes[1].set_ylabel('Estudios')
plt.tight_layout()

## 6. Ejemplos CT por clase

In [ ]:

def ct_lung_window(slice_2d: np.ndarray, window_min: float = -1000, window_max: float = 400) -> np.ndarray:
    """Normaliza un corte CT con ventana pulmonar para mejorar la visualizacion."""
    clipped = np.clip(slice_2d, window_min, window_max)
    return (clipped - window_min) / (window_max - window_min)


def find_ct_study_path(study_id: str, ct_class: str | None = None) -> Path | None:
    search_classes = [ct_class] if ct_class else ct_classes
    for cls in search_classes:
        if cls is None:
            continue
        for suffix in ['.nii', '.nii.gz']:
            candidate = ct_root / 'studies' / cls / f'{study_id}{suffix}'
            if candidate.exists():
                return candidate
    return None


def load_original_ct_slice(study_id: str, ct_class: str, slice_index: int | None = None) -> tuple[np.ndarray, int, int]:
    study_path = find_ct_study_path(study_id, ct_class)
    if study_path is None:
        raise FileNotFoundError(f'No se encontro el estudio {study_id} en {ct_class}')
    volume = nib.load(str(study_path)).get_fdata()
    total_slices = volume.shape[-1]
    if slice_index is None:
        slice_index = total_slices // 2
    slice_index = int(np.clip(slice_index, 0, total_slices - 1))
    image = np.rot90(volume[:, :, slice_index])
    return ct_lung_window(image), slice_index, total_slices


def load_png_image(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert('L'), dtype=np.float32) / 255.0


seg_metadata_path = ct_root / 'processed_segmentation_slices' / 'ct_segmentation_metadata.csv'
seg_df = pd.read_csv(seg_metadata_path)
seg_df['mask_px'] = seg_df['mask_path'].apply(lambda path: int((np.asarray(Image.open(path).convert('L')) > 0).sum()))
best_mask_row = seg_df.sort_values('mask_px', ascending=False).iloc[0]

representative_examples = {
    'CT-0': {'study_id': 'study_0001', 'slice_index': None, 'reason': 'ejemplo sin hallazgos evidentes'},
    'CT-1': {
        'study_id': best_mask_row['study_id'],
        'slice_index': int(best_mask_row['slice_index']),
        'image_path': Path(best_mask_row['image_path']),
        'mask_path': Path(best_mask_row['mask_path']),
        'reason': 'slice anotado con mayor area de mascara disponible',
    },
    'CT-2': {'study_id': 'study_0940', 'slice_index': 22, 'reason': 'ejemplo visual con opacidades moderadas'},
    'CT-3': {'study_id': 'study_1071', 'slice_index': 18, 'reason': 'ejemplo visual con afectacion extensa visible'},
    'CT-4': {'study_id': 'study_1109', 'slice_index': 20, 'reason': 'ejemplo visual de clase critica'},
}

fig, axes = plt.subplots(3, len(ct_classes), figsize=(16, 9), dpi=140)
selected_rows = []

for col, cls in enumerate(ct_classes):
    example = representative_examples[cls]
    study_id = example['study_id']

    if 'image_path' in example:
        image = load_png_image(example['image_path'])
        mask = np.asarray(Image.open(example['mask_path']).convert('L')) > 0
        slice_index = int(example['slice_index'])
        has_mask = True
    else:
        image, slice_index, _ = load_original_ct_slice(study_id, cls, example['slice_index'])
        mask = None
        has_mask = False

    selected_rows.append({
        'class': cls,
        'study_id': study_id,
        'slice_index': slice_index,
        'has_mask': has_mask,
        'reason': example['reason'],
    })

    axes[0, col].imshow(image, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'{cls}\n{study_id} | z={slice_index}', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].axis('off')
    if has_mask:
        axes[1, col].imshow(mask, cmap='gray')
        axes[1, col].set_title('Mascara lesion', fontsize=9)
    else:
        axes[1, col].imshow(np.zeros_like(image), cmap='gray', vmin=0, vmax=1)
        axes[1, col].text(0.5, 0.5, 'Sin mascara\ndisponible', transform=axes[1, col].transAxes, ha='center', va='center', color='white', fontsize=9)
        axes[1, col].set_title('Mascara lesion', fontsize=9)

    axes[2, col].imshow(image, cmap='gray', vmin=0, vmax=1)
    axes[2, col].axis('off')
    if has_mask:
        red_overlay = np.zeros((*mask.shape, 4), dtype=float)
        red_overlay[mask] = [1.0, 0.0, 0.0, 0.45]
        axes[2, col].imshow(red_overlay)
        axes[2, col].set_title('Overlay imagen + mascara', fontsize=9)
    else:
        axes[2, col].set_title('Overlay no disponible', fontsize=9)

fig.suptitle('Ejemplos CT representativos y mascaras disponibles', y=1.02)
fig.text(
    0.5,
    0.01,
    'Nota: las mascaras de infeccion disponibles pertenecen al subconjunto anotado; en esta copia local aparecen en estudios CT-1.',
    ha='center',
    fontsize=10,
)
plt.tight_layout(rect=[0, 0.04, 1, 0.98])

display(pd.DataFrame(selected_rows))


## 7. Conclusiones Fase 0

- CXR tiene imágenes y máscaras pulmonares para las 4 clases.
- Las máscaras CXR son de pulmón, no de lesión COVID; por tanto, solo permiten evaluar atención anatómica dentro/fuera del campo pulmonar.
- CT tiene 5 clases originales, pero CT-4 tiene solo 2 estudios; se fusiona con CT-3 como CT-3+ para obtener una tarea de 4 clases más defendible.
- Los splits de CT deben hacerse por `study_id`, nunca por slice, para evitar fuga de datos entre train/validation/test.